<a href="https://colab.research.google.com/github/lautarodibartolo-ae/t8002-representacion-de-datos/blob/main/clase-4-trabajo-final/04_trabajo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Clase 4 — El ejemplo completo

**Taller T8002 · Representación de datos**

Este notebook resuelve las **cinco piezas** de la sección 2 de la consigna sobre un caso chico: diez
préstamos de una biblioteca, en siete columnas, que terminan en cuatro tablas.

## No es una plantilla

Es el ejemplo de qué **forma** tiene un trabajo terminado: cuánto detalle lleva cada pieza y cómo se
ve una justificación escrita. Dos razones por las que no te sirve como molde:

- El caso es **más chico** que la tabla provista y que cualquiera que elijas.
- **No tiene columna multivalor**, así que su primera forma normal sale gratis. La tabla provista sí
  la tiene, y ese paso lo vas a tener que hacer solo. La clase 3, sección 4.2, lo resuelve completo.

## Cómo se usa

Ejecutá de arriba hacia abajo con `Shift + Enter`. No hay nada que completar: cada celda muestra un
resultado y el texto de abajo lo interpreta.

## Y no hace falta programar

El trabajo final **no se entrega como notebook**. Se entrega como un documento o una planilla, y no
lleva código. Acá hay código solo porque es la forma más corta de mostrar los datos y de comprobar
los resultados. Lo que se evalúa es el texto.

### Empezá por esta

In [ ]:
import pandas as pd
from itertools import combinations
from pathlib import Path

Path("crudo").mkdir(exist_ok=True)
print("pandas", pd.__version__)

In [ ]:
tabla = """socio_id,socio_nombre,isbn,titulo,editorial,editorial_ciudad,fecha_prestamo
104,"Aguirre, Nadia",978-950-04-1,El juguete rabioso,Losada,Buenos Aires,2026-04-06
104,"Aguirre, Nadia",978-987-11-2,Cuentos de la selva,Colihue,Buenos Aires,2026-04-06
104,"Aguirre, Nadia",978-987-57-5,El Aleph,Emecé,Barcelona,2026-05-11
217,"Molina, Hugo",978-950-04-1,El juguete rabioso,Losada,Buenos Aires,2026-04-14
217,"Molina, Hugo",978-950-04-3,La invención de Morel,Losada,Buenos Aires,2026-04-14
217,"Molina, Hugo",978-987-11-4,Facundo,Colihue,Buenos Aires,2026-05-04
338,"Paz, Cecilia",978-987-11-2,Cuentos de la selva,Colihue,Buenos Aires,2026-04-20
338,"Paz, Cecilia",978-987-57-5,El Aleph,Emecé,Barcelona,2026-04-20
338,"Paz, Cecilia",978-950-04-3,La invención de Morel,Losada,Buenos Aires,2026-05-18
338,"Paz, Cecilia",978-987-11-4,Facundo,Colihue,Buenos Aires,2026-05-18
"""

Path("crudo/prestamos.csv").write_text(tabla, encoding="utf-8")
df = pd.read_csv("crudo/prestamos.csv", dtype={"socio_id": str})

print(f"{len(df)} filas, {len(df.columns)} columnas")
df

Diez préstamos de una biblioteca. Tres socios, cinco libros, tres editoriales.

Leelo antes de seguir y hacete la pregunta del apunte: **¿qué hechos están escritos más de una
vez?**

---
# Pieza 1 — La tabla de origen y sus problemas

## La clave

Los cuatro pasos de la clase 1, sección 6.5.

In [ ]:
# Paso 1: ninguna columna sola alcanza.
resumen = pd.DataFrame({"valores_distintos": df.nunique(), "filas": len(df)})
resumen["¿sirve sola?"] = resumen["valores_distintos"] == len(df)
print(resumen.to_string())

# Paso 2: los pares.
print("\npares que identifican las 10 filas:")
for a, b in combinations(df.columns, 2):
    if df[[a, b]].drop_duplicates().shape[0] == len(df):
        print(f"   {a} + {b}")

**La clave es `socio_id` más `isbn`.** Es compuesta.

Aparecen seis pares, y cinco no son la clave. Dos motivos: `socio_nombre` hace de suplente de
`socio_id` y `titulo` hace de suplente de `isbn`, porque en estas diez filas cada socio y cada libro
tienen un nombre único. Y ningún libro se prestó dos veces el mismo día, así que `isbn` y `titulo`, emparejados con
`fecha_prestamo`, también distinguen las diez filas.

El par que sirve es `socio_id` más `isbn`: es el único donde las dos columnas identifican por regla.

**Paso 4, la pregunta que los datos no contestan:** ¿puede el mismo socio pedir el mismo libro dos
veces, en fechas distintas? En una biblioteca real, **sí**. Entonces la clave verdadera es el trío
`socio_id`, `isbn`, `fecha_prestamo`.

Y esa es una decisión de diseño que los datos no sugerían. Queda anotada, y la retomamos en la pieza
5.

## Los hechos repetidos, con su número

In [ ]:
for cols, que_es in [(["socio_id", "socio_nombre"], "el nombre de un socio"),
                     (["isbn", "titulo"], "el título de un libro"),
                     (["editorial", "editorial_ciudad"], "la ciudad de una editorial")]:
    distintos = df[cols].drop_duplicates().shape[0]
    print(f"{que_es}:")
    print(f"   hechos distintos: {distintos}")
    print(f"   filas que lo escriben: {len(df)}")
    print(f"   copias de más: {len(df) - distintos}")
    print()

Y el detalle de la peor:

In [ ]:
df.groupby(["editorial", "editorial_ciudad"]).size().rename("veces escrito")

**Las tres repeticiones, con su anomalía:**

| Qué se repite | Cuántas veces | Anomalía que produce |
|---|---|---|
| `socio_nombre`, 3 hechos en 10 filas | 7 copias de más | **Modificación.** Si Cecilia Paz cambia de apellido, hay que corregir 4 filas. Cualquier olvido deja dos nombres para el socio 338. |
| `titulo` y `editorial`, 5 hechos en 10 filas | 5 copias de más | **Inserción.** Un libro que la biblioteca compró y todavía nadie pidió no se puede cargar: no hay fila de préstamo donde escribirlo. |
| `editorial_ciudad`, 3 hechos en 10 filas | 7 copias de más | **Borrado.** Si se anulan los dos préstamos de Emecé, desaparece del archivo que Emecé está en Barcelona. |

Las tres repeticiones producen las tres anomalías: la clase 1, sección 3.3, dice que tienen la misma
causa y la misma cura. Acá elegí, para cada una, **la que más duele**. **No alcanza con decir "hay
redundancia": hay que decir qué anomalía se vuelve grave**, porque es lo que justifica separarla.

In [ ]:
# La anomalia de borrado, comprobada.
sin_emece = df[df["editorial"] != "Emecé"]
print(f"filas antes: {len(df)}   después: {len(sin_emece)}")
print("¿sigue existiendo Barcelona en el archivo?",
      "sí" if "Barcelona" in sin_emece["editorial_ciudad"].values else "NO")

---
# Pieza 2 — El diseño conceptual

Cuatro entidades. Para cada una, las tres preguntas de la clase 2, sección 3.3:

| Candidata | ¿Propiedades propias? | ¿Existe sola? | ¿Hay que administrar su lista? | Veredicto |
|---|---|---|---|---|
| socio | sí, el nombre | sí, un socio nuevo sin préstamos | sí, se dan de alta socios | **entidad** |
| libro | sí, el título y la editorial | sí, un libro comprado sin prestar | sí, entra cada compra | **entidad** |
| editorial | sí, la ciudad | sí | sí, se agregan editoriales | **entidad** |
| préstamo | sí, la fecha | no, necesita socio y libro | no | **entidad** (el vínculo) |

`editorial_ciudad` no es entidad: es un atributo de `editorial`. Si la ciudad tuviera datos propios
—un país, una zona— habría que volver a preguntar.

El préstamo es el único que sale con un solo sí, y alcanza: la regla corta de la clase 2 dice que
basta con tener datos propios. La fecha es del préstamo y de nada más, y por eso el vínculo entre
socio y libro es una entidad y no una línea.

## Las cardinalidades, en los dos sentidos

- Un libro lo publica **una sola** editorial; una editorial publica **muchos** libros.
- Un préstamo corresponde a **un solo** libro; un libro tiene **muchos** préstamos.
- Un préstamo lo hace **un solo** socio; un socio hace **muchos** préstamos.

Y la participación:

- Un libro **tiene que** tener editorial: obligatoria.
- Una editorial **puede** no tener ningún libro cargado todavía: opcional. Es la que resuelve la
  anomalía de inserción.

> **Cuidado con lo que sugieren diez filas.** Acá cada socio tiene entre 3 y 4 préstamos, así que el
> vínculo se ve claramente uno a muchos. Pero si la muestra tuviera un préstamo por socio, parecería
> uno a uno, y seguiría siendo uno a muchos. La cardinalidad se decide por la regla de la
> biblioteca, no contando filas.

## El diagrama

```
┌──────────────┐ 1      N ┌──────────────┐ 1      N ┌──────────────┐
│ EDITORIAL    │ ──────── │ LIBRO        │ ──────── │ PRESTAMO     │
│ nombre       │ publica  │ isbn         │  tiene   │ fecha        │
│ ciudad       │          │ titulo       │          └──────────────┘
└──────────────┘          └──────────────┘
                                                           │ N
                                                           │ hace
                                                           │ 1
                                                    ┌──────────────┐
                                                    │ SOCIO        │
                                                    │ id           │
                                                    │ nombre       │
                                                    └──────────────┘
```

El primer atributo de cada caja es el que identifica. En papel se subraya; acá no se puede, así que
va primero. La notación no se evalúa: esto mismo dibujado a mano y fotografiado vale igual.

---
# Pieza 3 — Las dependencias funcionales

Cinco flechas, agrupadas por forma.

Para buscarlas usamos `socio_id` más `isbn`, que es la clave **de estas diez filas**. El diseño
final va a usar el trío con la fecha, por la razón de la pieza 1, y eso no cambia ninguna de las
cinco flechas: agregar una columna a la izquierda nunca rompe una dependencia.

In [ ]:
def determina(datos, izquierda, derecha):
    izquierda = izquierda if isinstance(izquierda, list) else [izquierda]
    conteo = datos.groupby(izquierda, dropna=False)[derecha].nunique(dropna=False)
    return bool((conteo <= 1).all())

CLAVE = ["socio_id", "isbn"]

print("TOTALES (dependen de la clave entera)")
for col in df.columns:
    if col not in CLAVE and determina(df, CLAVE, col):
        print(f"   socio_id , isbn        ->  {col}")

print("\nPARCIALES (dependen de media clave)")
for mitad in CLAVE:
    for col in df.columns:
        if col not in CLAVE and determina(df, mitad, col):
            print(f"   {mitad:22} ->  {col}")

print("\nTRANSITIVAS (el lado izquierdo no es clave ni parte de clave)")
for col in ["editorial_ciudad"]:
    if determina(df, "editorial", col):
        print(f"   editorial              ->  {col}")

Fijate que la lista de parciales que imprimió es más larga que la que sirve. Salieron cuatro:
`socio_id → socio_nombre`, `isbn → titulo`, `isbn → editorial` e `isbn → editorial_ciudad`. Las tres
primeras son reales. La cuarta no es una casualidad: es **derivada**, porque el ISBN fija la
editorial y la editorial fija su ciudad. Por eso no se lista aparte: se resuelve en el paso a 3FN.

Es la trampa de la clase 3, sección 2.4. **Los datos descartan dependencias; no las confirman.**
Estas se descartan preguntando por la regla, y más adelante en este notebook la trampa va a aparecer
otra vez, adentro de la propia herramienta de verificación.

La lista buena, después de filtrar las casualidades:

```
TOTALES
    socio_id, isbn   →  fecha_prestamo

PARCIALES
    socio_id          →  socio_nombre
    isbn              →  titulo
    isbn              →  editorial

TRANSITIVAS
    editorial         →  editorial_ciudad
```

**Cinco flechas**, y las cuatro que no son totales son la lista concreta de lo que hay que
separar.

De dónde sale cada una, que es lo que la consigna pide justificar:

- `socio_id → socio_nombre`: el nombre lo fija la persona y no el préstamo. Dos préstamos del mismo
  socio no pueden tener nombres distintos.
- `isbn → titulo, editorial`: el ISBN identifica una edición concreta. Dos ejemplares con el mismo
  ISBN son el mismo libro, de la misma editorial.
- `editorial → editorial_ciudad`: la ciudad la fija la casa editora y no el libro. Dos libros de
  Losada no pueden tener ciudades distintas.

---
# Pieza 4 — El diseño en tercera forma normal

## Paso a 1FN

In [ ]:
separadores = df.apply(lambda col: col.astype(str).str.contains(r"[;,/]").sum())
print("celdas con separadores adentro, por columna:")
print(separadores[separadores > 0].to_string())

Aparece `socio_nombre` con comas, y **no es una violación de 1FN**: `"Paz, Cecilia"` es un solo
valor, un apellido y un nombre. La coma está adentro del dato, no separa dos datos.

Ninguna otra columna tiene separadores. **La tabla ya está en primera forma normal**, y no hay nada
que hacer en este paso.

Es el paso que en la tabla provista sí hay que hacer, por la columna `materiales`.

## Paso a 2FN: se van las dependencias parciales

In [ ]:
socio = (df[["socio_id", "socio_nombre"]].drop_duplicates()
         .rename(columns={"socio_id": "id", "socio_nombre": "nombre"})
         .sort_values("id").reset_index(drop=True))

libro_2fn = (df[["isbn", "titulo", "editorial", "editorial_ciudad"]].drop_duplicates()
             .sort_values("isbn").reset_index(drop=True))

prestamo = (df[["socio_id", "isbn", "fecha_prestamo"]]
            .sort_values(["socio_id", "isbn"]).reset_index(drop=True))

print(f"socio:     {len(socio)} filas")
print(f"libro:     {len(libro_2fn)} filas")
print(f"prestamo:  {len(prestamo)} filas")
print()
print(socio.to_string(index=False))

Tres tablas. El nombre de cada socio quedó escrito **una vez**, no cuatro.

## Paso a 3FN: se va la dependencia transitiva

In [ ]:
editorial = (libro_2fn[["editorial", "editorial_ciudad"]].drop_duplicates()
             .rename(columns={"editorial": "nombre", "editorial_ciudad": "ciudad"})
             .sort_values("nombre").reset_index(drop=True))

libro = libro_2fn.drop(columns=["editorial_ciudad"]).rename(
    columns={"editorial": "editorial_nombre"})

print("editorial")
print(editorial.to_string(index=False))
print("\nlibro")
print(libro.to_string(index=False))

**Cuatro tablas.** El diseño final:

| Tabla | Filas | Clave primaria | Claves foráneas |
|---|---|---|---|
| `editorial` | 3 | `nombre` | — |
| `libro` | 5 | `isbn` | `editorial_nombre` → `editorial.nombre` |
| `socio` | 3 | `id` | — |
| `prestamo` | 10 | `socio_id` + `isbn` + `fecha_prestamo` | `socio_id` → `socio.id`, `isbn` → `libro.isbn` |

Y el recorrido, que es lo que la consigna pide en la pieza 4:

| Paso | Qué se resolvió |
|---|---|
| a 1FN | nada: la tabla ya estaba, no había celdas multivalor |
| a 2FN | `socio_id → socio_nombre` y `isbn → titulo, editorial`, que dependían de media clave |
| a 3FN | `editorial → editorial_ciudad`, que era una cadena de dos pasos |

La clave de `prestamo` lleva la fecha por la decisión de la pieza 1: sobre estas diez filas el par
alcanzaría, pero la regla de la biblioteca permite pedir el mismo libro dos veces.

## Las dos pruebas de la sección 6 de la consigna

In [ ]:
# Prueba 1: ninguna tabla tiene dependencias parciales ni transitivas.
def revisar(tabla, nombre, clave):
    problemas = []
    no_clave = [c for c in tabla.columns if c not in clave]
    if len(clave) > 1:
        for mitad in clave:
            for col in no_clave:
                if determina(tabla, mitad, col):
                    problemas.append(f"parcial: {mitad} -> {col}")
    for a in no_clave:
        for b in no_clave:
            if a != b and determina(tabla, a, b):
                problemas.append(f"transitiva: {a} -> {b}")
    print(f"{nombre:12} {'LIMPIA' if not problemas else problemas}")

revisar(editorial, "editorial", ["nombre"])
revisar(libro,     "libro",     ["isbn"])
revisar(socio,     "socio",     ["id"])
revisar(prestamo,  "prestamo",  ["socio_id", "isbn", "fecha_prestamo"])

# En `prestamo` las tres columnas son la clave, asi que no queda ninguna que
# pueda depender de otra: ahi LIMPIA quiere decir que no hay nada que revisar.

Tres limpias, y una marcada. Y la marcada **no es un error del diseño**: es el falso positivo de
la clase 3, sección 2.4, apareciendo dentro de la propia herramienta de verificación.

`titulo → editorial_nombre` pasa la prueba porque los cinco libros tienen títulos distintos. Con
cinco filas, el título identifica un libro tanto como el ISBN: es **otra clave candidata**. Y una
dependencia que sale de una clave candidata no es transitiva, así que no viola la tercera forma
normal.

Cómo se confirma que es un falso positivo: preguntando por la regla. ¿Puede haber dos libros con el
mismo título? Sí, dos ediciones distintas de *Facundo*, de editoriales distintas, con ISBN distintos.
Entonces el título **no** identifica un libro y la dependencia se cae.

> **La lección, y es la del taller entero:** ninguna verificación automática reemplaza la pregunta
> por la regla del negocio. La herramienta te dice dónde mirar, no qué decidir. En tu trabajo final
> vas a ver candidatas así, y tenés que descartarlas a mano.

In [ ]:
# Prueba 2: cruzando las cuatro tablas vuelve la tabla original.
vuelta = (prestamo
    .merge(socio.rename(columns={"id": "socio_id", "nombre": "socio_nombre"}), on="socio_id")
    .merge(libro, on="isbn")
    .merge(editorial.rename(columns={"nombre": "editorial_nombre",
                                     "ciudad": "editorial_ciudad"}), on="editorial_nombre")
    .rename(columns={"editorial_nombre": "editorial", "fecha": "fecha_prestamo"}))

izq = vuelta[list(df.columns)].sort_values(["socio_id", "isbn"]).reset_index(drop=True)
der = df.sort_values(["socio_id", "isbn"]).reset_index(drop=True)

print(f"filas: {len(izq)} de {len(der)}")
print("¿idénticas, celda por celda?", izq.equals(der))

`True`. **Las dos pruebas pasan**, y esa es la señal de que el diseño está bien.

La segunda es la que más se saltea, y es la única que detecta una descomposición mal hecha. En el
trabajo final se puede hacer a mano: elegí tres filas de la tabla original y seguilas por tus tablas
nuevas, cruzando por las claves.

---
# Pieza 5 — La justificación

Esto es texto, y es lo que más pesa en la devolución. Media carilla alcanza.

---

**Qué anomalía dejó de ser posible.** La de modificación desapareció por construcción. El nombre del
socio 338 estaba en 4 filas y ahora está en 1: no existe un estado en el que dos filas discrepen,
porque hay una sola celda donde escribir. Lo mismo con la ciudad de cada editorial, que pasó de 10
celdas a 3. La de inserción también: un libro comprado y todavía no prestado ahora se carga como una
fila de `libro`, sin inventar un préstamo que no existió, y una editorial nueva se carga sin libros.
Y la de borrado: anular los dos préstamos de Emecé ya no borra que Emecé está en Barcelona.

**Qué se volvió más incómodo.** Ver un préstamo completo, con el nombre del socio y el título del
libro, ahora exige cruzar cuatro tablas en vez de mirar una fila. Con diez préstamos es claramente
peor que la planilla original. La salida es una vista, que muestra la tabla ancha sin duplicar el
dato, y en una base de datos se define una vez. Sin vista, cualquier consulta simple pasa a ser un
cruce de cuatro tablas, y para una biblioteca chica ese costo puede no valer la pena: si el archivo
solo se lee y nadie carga nada nuevo, la tabla plana está bien.

**La decisión que no era la obvia.** Los diez préstamos indicaban que `socio_id` más `isbn` alcanzaba
como clave: las diez combinaciones son distintas. Igual definí la clave como el trío `socio_id`,
`isbn`, `fecha_prestamo`, porque un socio puede pedir el mismo libro dos veces en fechas distintas, y
con la clave de dos columnas el segundo préstamo sería rechazado como duplicado. Los datos no
mostraban ese caso porque en diez préstamos no apareció, no porque sea imposible. Alternativa que
descarté: una clave sustituta `id_prestamo`. Es más cómoda para referirse a un préstamo desde otra
tabla, y la descarté porque hoy ninguna tabla necesita referirse a un préstamo, y porque una clave
sustituta habría dejado sin declarar la regla de que no puede haber dos préstamos iguales el mismo
día. Lo que se pierde con mi decisión: si mañana hace falta una tabla de devoluciones, va a tener que
arrastrar tres columnas para referirse al préstamo.

---

Fijate qué hace ese último párrafo, porque es la diferencia entre una justificación que vale poco y
una que vale mucho: **nombra la alternativa que descartó, dice por qué, y dice qué se pierde.** Es lo
mismo que pedía el trabajo final del T8001.

---
# Cierre

Las cinco piezas, y cuánto ocupó cada una:

| Pieza | Qué es | Tamaño en este ejemplo |
|---|---|---|
| 1 | La tabla y sus problemas | la clave, y tres repeticiones con su anomalía |
| 2 | El diseño conceptual | cuatro entidades, tres vínculos, un diagrama |
| 3 | Las dependencias funcionales | cinco flechas, agrupadas, con su regla |
| 4 | El diseño en 3FN | cuatro tablas, el recorrido y las dos pruebas |
| 5 | La justificación | tres párrafos |

Ese es el tamaño. **No hace falta más**, y las piezas 1 y 5 son las que más pesan.

Ahora te toca, sobre `capacitaciones.csv` o sobre tu propia tabla. Las dos diferencias con este
ejemplo: la tabla es más grande, y **tiene una columna multivalor**, así que el paso a 1FN esta vez
hay que hacerlo. Está resuelto en la clase 3, sección 4.2.

La consigna completa está en el documento de esta clase.